ដើម្បីដំណើរការប្រអប់កំណត់ត្រាខាងក្រោម បើអ្នកមិនបានធ្វើវានៅឡើយទេ អ្នកត្រូវតែ triển khai ម៉ូដែលដែលប្រើប្រាស់ `text-embedding-ada-002` ជាម៉ូដែលមូលដ្ឋាន ហើយកំណត់ឈ្មោះ triển khai នៅក្នុងឯកសារ .env ជា `AZURE_OPENAI_EMBEDDINGS_ENDPOINT`


In [ ]:
import os
import pandas as pd
import numpy as np
from openai import AzureOpenAI
from dotenv import load_dotenv

load_dotenv()

client = AzureOpenAI(
  api_key=os.environ['AZURE_OPENAI_API_KEY'],  # this is also the default, it can be omitted
  api_version = "2024-10-21",
  azure_endpoint = os.environ['AZURE_OPENAI_ENDPOINT']
  )

model = os.environ['AZURE_OPENAI_EMBEDDINGS_DEPLOYMENT']

SIMILARITIES_RESULTS_THRESHOLD = 0.75
DATASET_NAME = "../embedding_index_3m.json"


បន្តទៅ ខណៈនេះ យើងនឹងផ្ទុក ឯកសារ បន្ធូរបន្ទាញ់ Embedding Index ទៅក្នុង Dataframe របស់ Pandas។ ឯកសារ Embedding Index ត្រូវបានរក្សាទុកនៅក្នុងឯកសារ JSON ដែលមានឈ្មោះ `embedding_index_3m.json`។ ឯកសារ Embedding Index មាន Embeddings សម្រាប់រាល់អក្សរសម្រាប់វីដេអូ YouTube រហូតដល់ចុងខែតុលា ឆ្នាំ ២០២៣។


In [ ]:
def load_dataset(source: str) -> pd.core.frame.DataFrame:
    # Load the video session index
    pd_vectors = pd.read_json(source)
    return pd_vectors.drop(columns=["text"], errors="ignore").fillna("")

បន្ទាប់មក យើងត្រូវបង្កើតមុខងារ​មួយ​ឈ្មោះថា `get_videos` ដែលនឹងស្វែងរកក្នុងតារាង Embedding សម្រាប់សំនួរ។ មុខងារនេះនឹងត្រឡប់វិដេអូ ៥ សិប្ដាដែលស្រដៀងគ្នាចម្បងជាងគេសម្រាប់សំនួរ។ មុខងារនេះដំណើរការដូចតទៅ៖

១. ជាក់លាក់ជាងមុន ចម្លងតារាង Embedding ត្រូវបានបង្កើត។
២. បន្ទាប់មក Embedding សម្រាប់សំនួរត្រូវបានគណនាដោយប្រើ OpenAI Embedding API។
៣. បន្ទាប់មក បង្កើតជួរឈរថ្មីមួយនៅក្នុងតារាង Embedding ដែលឈ្មោះ `similarity`។ ជួរឈរ `similarity` នេះមានតម្លៃស្រដៀងគ្នា(costlike similarity) រវាង Embedding របស់សំនួរនិង Embedding សម្រាប់មុខវិដេអូទាំងអស់។
៤. បន្ទាប់មក តារាង Embedding ត្រូវបានចម្រាញ់តាមជួរឈរ `similarity`។ តារាង Embedding ត្រូវបានចម្រាញ់ដើម្បីរួមមានតែវិដេអូដែលមានតម្លៃស្រដៀងគ្នា(costlike similarity) ធំជាង ឬស្មើ ០.៧៥។
៥. ចុងក្រោយ តារាង Embedding ត្រូវបានតម្រៀបតាមជួរឈរ `similarity` ហើយវិដេអូ ៥ ខាងលើត្រូវបានត្រឡប់មកវិញ។


In [ ]:
def cosine_similarity(a, b):
    if len(a) > len(b):
        b = np.pad(b, (0, len(a) - len(b)), 'constant')
    elif len(b) > len(a):
        a = np.pad(a, (0, len(b) - len(a)), 'constant')
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def get_videos(
    query: str, dataset: pd.core.frame.DataFrame, rows: int
) -> pd.core.frame.DataFrame:
    # create a copy of the dataset
    video_vectors = dataset.copy()

    # get the embeddings for the query    
    query_embeddings = client.embeddings.create(input=query, model=model).data[0].embedding

    # create a new column with the calculated similarity for each row
    video_vectors["similarity"] = video_vectors["ada_v2"].apply(
        lambda x: cosine_similarity(np.array(query_embeddings), np.array(x))
    )

    # filter the videos by similarity
    mask = video_vectors["similarity"] >= SIMILARITIES_RESULTS_THRESHOLD
    video_vectors = video_vectors[mask].copy()

    # sort the videos by similarity
    video_vectors = video_vectors.sort_values(by="similarity", ascending=False).head(
        rows
    )

    # return the top rows
    return video_vectors.head(rows)

មុខងារនេះយ៉ាងងាយស្រួលណាស់ វាពិតជាបង្ហាញលទ្ធផលនៃការស្វែងរក។


In [ ]:
def display_results(videos: pd.core.frame.DataFrame, query: str):
    def _gen_yt_url(video_id: str, seconds: int) -> str:
        """convert time in format 00:00:00 to seconds"""
        return f"https://youtu.be/{video_id}?t={seconds}"

    print(f"\nVideos similar to '{query}':")
    for _, row in videos.iterrows():
        youtube_url = _gen_yt_url(row["videoId"], row["seconds"])
        print(f" - {row['title']}")
        print(f"   Summary: {' '.join(row['summary'].split()[:15])}...")
        print(f"   YouTube: {youtube_url}")
        print(f"   Similarity: {row['similarity']}")
        print(f"   Speakers: {row['speaker']}")

1. ជាមុនសិន ធាតុបញ្ចូល Embedding Index ត្រូវបានផ្ទុកចូលក្នុង Pandas Dataframe។
2. បន្ទាប់មក អ្នកប្រើត្រូវបានអោយបញ្ចូលសំណួរ។
3. បន្ទាប់មកហៅមុខងារ `get_videos` ដើម្បីស្វែងរក Embedding Index សម្រាប់សំណួរនោះ។
4. ចុងបញ្ចប់ហៅមុខងារ `display_results` ដើម្បីបង្ហាញលទ្ធផលទៅអ្នកប្រើ។
5. បន្ទាប់មកអ្នកប្រើត្រូវបានអោយបញ្ចូលសំណួរថ្មីម្តងទៀត។ ដំណើរការនេះបន្តរហូតដល់អ្នកប្រើបញ្ចូល `exit`។

![](../../../../translated_images/km/notebook-search.1e320b9c7fcbb0bc.webp)

អ្នកនឹងត្រូវបានអោយបញ្ចូលសំណួរ។ សូមបញ្ចូលសំណួរហើយចុច enter។កម្មវិធីនឹងត្រឡប់មកបញ្ជីវីដេអូដែលពាក់ព័ន្ធនឹងសំណួរនេះ។កម្មវិធីក៏នឹងផ្ដល់តំណទៅកន្លែងនៅក្នុងវីដេអូដែលមានចម្លើយសម្រាប់សំណួរនោះ។

ទីនេះជាសំណួរខ្លះៗសម្រាប់សាកល្បង៖

- Azure Machine Learning ជាអ្វី?
- តើបណ្ដាញសរសៃប្រសាទ convolutional ធ្វើការយ៉ាងដូចម្តេច?
- បណ្ដាញសរសៃប្រសាទតើមានន័យអ្វី?
- តើខ្ញុំអាចប្រើ Jupyter Notebooks ជាមួយ Azure Machine Learning ទេ?
- ONNX ជាអ្វី?


In [ ]:
pd_vectors = load_dataset(DATASET_NAME)

# get user query from input
while True:
    query = input("Enter a query: ")
    if query == "exit":
        break
    videos = get_videos(query, pd_vectors, 5)
    display_results(videos, query)

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ការបដិសេធ**:
ឯកសារនេះត្រូវបានបម្លែងភាសា ដោយប្រើសេវាបម្លែងភាសា AI [Co-op Translator](https://github.com/Azure/co-op-translator)។ ទោះយើងខ្ញុំមានក្តីប្រាថ្នាឱ្យបានច្បាស់លាស់ តែសូមយល់ដឹងថាការបម្លែងដោយស្វ័យប្រវត្តិក៏អាចមានកំហុសឬភាពមិនត្រឹមត្រូវ។ ឯកសារដើមជាភាសាទីតាំងគួរត្រូវបានគេប្រើជាប្រភពច្បាស់លាស់។ សម្រាប់ព័ត៌មានសំខាន់ៗ សូមណែនាំឱ្យប្រើប្រាស់ការប្រែដោយមនុស្សជំនាញ។ យើងខ្ញុំមិនទទួលខុសត្រូវចំពោះការយល់ច្រឡំ ឬការបកស្រាយខុសបន្ទាប់ពីការប្រើប្រាស់ការបម្លែងនេះនោះទេ។
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
